# Text → gloss engines: rules v1 / v2, the team's T5 hybrid, and the guarded hybrid

**What this notebook does.** Diagnostic (no training), TODO §13 Phase 2. It scores every
English → gloss engine in `sb.synthesize.gloss` on two test sets:

| engine | what |
|---|---|
| `rules_v1` | the team's rule engine, frozen (T5's training input) |
| `rules_v2` | the fixed engine (ME, conjunctions, particles, time fronting, negation) |
| `hybrid_team` | rules_v1 → T5 with the team's exact `generate()` (T5 trusted) |
| `hybrid_guarded` | rules_v1 → T5 with repetition control → meaning guard; rejected → rules_v2 |
| `t5_recorded` | the team's **recorded** T5 outputs on team30 (their notebook's batch run) |
| `guard_on_recorded` | the guard applied to those recorded outputs, rules_v2 as fallback |

The last two rows need no checkpoint, so the guard is measured **now**. The hybrids fall
back to rules until the T5 checkpoint is at `t5.model_dir` (config).

**Test sets**
- **team30** (§1): the team's 30 sentences against **draft** references. `rules_v2` was
  written to the same conventions by the same author, so its team30 score is a
  **development** number, not a test result.
- **ASLG-PC12 test** (§3): a seeded sample of the fixed test split, scored with
  `DESC-`/`X-` markers stripped. Its glosses are synthetic Europarl rule output, so
  this measures agreement with *that* rule system.

**Metrics** (`sb.synthesize.metrics`): BLEU-4, chrF, ROUGE-L, METEOR, gloss WER (S/D/I),
exact match, plus lexicon coverage (`sb.synthesize.lexicon`).

| artifact | path |
|---|---|
| team30 per-sentence outputs + scores | `data/cache/synthesis/text2gloss/team30_outputs.csv` |
| team30 corpus scores | `data/cache/synthesis/text2gloss/team30_scores.csv` |
| ASLG outputs per engine | `data/cache/synthesis/text2gloss/aslg_test_<engine>.parquet` |
| ASLG corpus scores | `data/cache/synthesis/text2gloss/aslg_scores.csv` |
| coverage | `data/cache/synthesis/text2gloss/coverage.csv` |

**Resumable:** ASLG outputs are cached per engine and skipped if present (delete a file to
recompute; a T5 checkpoint arriving means deleting the two `hybrid_*` files).

## Setup

In [1]:
# ============================================================
# Imports, config, paths
# ============================================================
import json

import pandas as pd

from sb.core.paths import CACHE_DIR, EXPERIMENTS_DIR, ROOT_DIR
from sb.synthesize import data as D
from sb.synthesize import evalsets, lexicon, metrics
from sb.synthesize.gloss import GlossEngine, hybrid, t5

CONFIG_DIR = EXPERIMENTS_DIR / "synthesis" / "configs"
CONFIG = json.loads((CONFIG_DIR / "aslg.text2gloss.json").read_text(encoding="utf-8"))
T5_DIR = ROOT_DIR / CONFIG["t5"]["model_dir"]
OUT = CACHE_DIR / "synthesis" / "text2gloss"
OUT.mkdir(parents=True, exist_ok=True)

print(f"ASLG-PC12 raw : {D.ASLG_RAW}")
print(f"splits        : {D.ASLG_CACHE / 'splits' / CONFIG['splits']['version']}")
print(f"WLASL index   : {lexicon.WLASL_JSON}")
print(f"T5 checkpoint : {T5_DIR} -> {'FOUND' if t5.available(T5_DIR) else 'missing (hybrids fall back to rules)'}")
print(f"results       : {OUT}")

ASLG-PC12 raw : C:\Users\user2\repository\data\raw\aslg_pc12
splits        : C:\Users\user2\repository\data\cache\aslg_pc12\splits\v1
WLASL index   : C:\Users\user2\repository\data\raw\wlasl\WLASL_v0.3.json
T5 checkpoint : C:\Users\user2\repository\data\external\t5-text2gloss\thesis_hybrid_dataset1\model -> missing (hybrids fall back to rules)
results       : C:\Users\user2\repository\data\cache\synthesis\text2gloss


## 1. team30: every engine vs the draft references

In [2]:
# ============================================================
# team30: outputs + per-sentence and corpus scores
# ============================================================
from sb.synthesize.gloss import rules_v1, rules_v2

t30 = evalsets.load("team30", CONFIG["eval"]["team30_version"])
sents = t30.english.tolist()
outs = {name: [r["gloss"] for r in GlossEngine(name, CONFIG["t5"]).gloss_sentences(sents)]
        for name in CONFIG["eval"]["engines"]}
outs["t5_recorded"] = t30.hybrid_recorded.tolist()
v1 = [rules_v1.convert(s) for s in sents]
v2 = [rules_v2.convert(s) for s in sents]
outs["guard_on_recorded"] = [hybrid.combine(s, g1, t, g2, min_content_recall=CONFIG["t5"]["guard"]["min_content_recall"]).gloss
                             for s, g1, t, g2 in zip(sents, v1, t30.hybrid_recorded, v2)]

refs = [metrics.normalize_gloss(r) for r in t30.reference_draft]
rows, scores = [], {}
for name, hyp in outs.items():
    hyp = [metrics.normalize_gloss(h) for h in hyp]
    scores[name] = metrics.corpus_scores(refs, hyp)
    for i, (r, h) in enumerate(zip(refs, hyp)):
        rows.append({"id": i + 1, "engine": name, "english": sents[i], "reference": r, "gloss": h,
                     **metrics.sentence_scores(r, h)})
per = pd.DataFrame(rows)
per.to_csv(OUT / "team30_outputs.csv", index=False)
tab = pd.DataFrame(scores).T[["bleu4", "chrf", "rouge_l", "meteor", "gloss_wer", "sub", "del", "ins", "exact"]]
tab.to_csv(OUT / "team30_scores.csv")
print("hybrid engines used T5:", t5.available(T5_DIR), "(False -> hybrid_team == rules_v1, hybrid_guarded == rules_v2)")
tab.sort_values("gloss_wer").round(3)

hybrid engines used T5: False (False -> hybrid_team == rules_v1, hybrid_guarded == rules_v2)


,bleu4,chrf,rouge_l,meteor,gloss_wer,sub,del,ins,exact
rules_v2,96.749,98.154,98.488,97.677,0.020,0.007,0.007,0.007,0.900
hybrid_guarded,96.749,98.154,98.488,97.677,0.020,0.007,0.007,0.007,0.900
guard_on_recorded,66.254,85.538,87.415,89.952,0.224,0.079,0.059,0.086,0.400
t5_recorded,36.041,72.240,76.048,75.881,0.414,0.178,0.112,0.125,0.067
hybrid_team,39.292,70.152,70.870,73.752,0.428,0.263,0.112,0.053,0.167
rules_v1,39.292,70.152,70.870,73.752,0.428,0.263,0.112,0.053,0.167


## 2. The guard on the team's recorded T5 outputs

For each sentence: the team's T5 output, whether the guard kept it, and why not. The
audit's meaning-changing cases (#19 you → HE, #25 her → ME, #20 repetition, dropped
subjects #2/#4/#12) should all be rejected, and T5's wins (#1, #26, #29) kept.

In [3]:
# ============================================================
# Guard decisions on recorded T5
# ============================================================
from sb.synthesize.gloss import rules_v1

t30 = evalsets.load("team30", CONFIG["eval"]["team30_version"])
dec = []
for _, r in t30.iterrows():
    g = hybrid.guard(r.english, rules_v1.convert(r.english), r.hybrid_recorded,
                     min_content_recall=CONFIG["t5"]["guard"]["min_content_recall"])
    dec.append({"id": r.id, "english": r.english, "t5_recorded": r.hybrid_recorded,
                "kept": g.accepted, "reasons": ", ".join(g.reasons)})
dec = pd.DataFrame(dec)
print(f"kept {dec.kept.sum()}/{len(dec)}; rejection reasons:",
      pd.Series([x.split(":")[0] for r in dec.reasons if r for x in r.split(", ")]).value_counts().to_dict())
with pd.option_context("display.max_colwidth", 60, "display.width", 200):
    display(dec)

kept 19/30; rejection reasons: {'person': 10, 'invented': 4, 'repetition': 3, 'content': 1}


,id,english,t5_recorded,kept,reasons
0,1,I am going home now.,ME NOW GO HOME,True,
1,2,She likes coffee.,LIKE COFFEE,False,person
2,3,My brother works at a hospital.,MY BROTHER WORK HOSPITAL,True,
3,4,We bought a new car yesterday.,YESTERDAY BUY NEW CAR,False,person
4,5,The children are playing outside.,CHILDREN PLAY OUTSIDE,True,
5,6,I will visit my mother tomorrow.,ME FUTURE VISIT MY MOTHER TOMORROW,True,
6,7,John went to the market this morning.,JOHN GO TO MARKET THIS MORNING,True,
7,8,We have a meeting at three o'clock.,WE HAVE MEET THREE O'CLOCK,True,
8,9,She will stay in Dhaka for two days.,FUTURE STAY DHAKA FOR TWO DAY,False,person
9,10,My appointment is on Friday morning.,APPOINTMENT FRIDAY MORNING,False,person


## 3. ASLG-PC12 test sample

A seeded sample of the fixed test split (size in the config), `DESC-`/`X-` stripped from
references before scoring. One cached parquet per engine.

In [4]:
# ============================================================
# ASLG-PC12 test: outputs per engine (cached) + corpus scores
# ============================================================
from tqdm.auto import tqdm

ev = CONFIG["eval"]
test = D.aslg_splits(CONFIG["splits"]["fractions"], CONFIG["splits"]["seed"], CONFIG["splits"]["version"])["test"]
test = test.sample(min(ev["aslg_test_sample"], len(test)), random_state=ev["aslg_sample_seed"]).reset_index(drop=True)
refs = [metrics.normalize_gloss(g, style="aslg") for g in test.gloss]
scores = {}
bar = tqdm(ev["engines"], desc="ASLG engines")
for name in bar:
    path = OUT / f"aslg_test_{name}.parquet"
    if not path.exists():
        bar.set_postfix_str(name)
        eng = GlossEngine(name, CONFIG["t5"])
        hyp = [r["gloss"] for r in eng.gloss_sentences(test.text.tolist())]
        pd.DataFrame({"text": test.text, "reference": refs, "gloss": hyp}).to_parquet(path, index=False)
    got = pd.read_parquet(path)
    scores[name] = metrics.corpus_scores(got.reference.tolist(), [metrics.normalize_gloss(h) for h in got.gloss])
tab = pd.DataFrame(scores).T[["n", "bleu4", "chrf", "rouge_l", "meteor", "gloss_wer", "sub", "del", "ins", "exact"]]
tab.to_csv(OUT / "aslg_scores.csv")
tab.round(3)

ASLG engines:   0%|          | 0/4 [00:00<?, ?it/s]

,n,bleu4,chrf,rouge_l,meteor,gloss_wer,sub,del,ins,exact
rules_v1,2000.0,26.283,71.031,76.591,63.607,0.347,0.059,0.284,0.004,0.024
rules_v2,2000.0,36.387,76.961,78.696,70.038,0.303,0.085,0.212,0.007,0.032
hybrid_team,2000.0,26.283,71.031,76.591,63.607,0.347,0.059,0.284,0.004,0.024
hybrid_guarded,2000.0,36.387,76.961,78.696,70.038,0.303,0.085,0.212,0.007,0.032


## 4. Lexicon coverage

Share of gloss units (multi-word WLASL entries matched first) that are real signs (WLASL
or GISLR) vs fingerspelled, and share of sentences our 250-sign recognizer could fully
sign back. Reads the §1 and §3 outputs from disk.

In [5]:
# ============================================================
# Coverage per engine (team30 + ASLG sample)
# ============================================================
lex = lexicon.Lexicon()
per = pd.read_csv(OUT / "team30_outputs.csv", keep_default_na=False)
cov = {}
for name, g in per.groupby("engine", sort=False):
    cov[("team30", name)] = lex.coverage(g.gloss.tolist())
cov[("team30", "reference_draft")] = lex.coverage(per[per.engine == per.engine.iloc[0]].reference.tolist())
for p in sorted(OUT.glob("aslg_test_*.parquet")):
    cov[("aslg", p.stem.removeprefix("aslg_test_"))] = lex.coverage(pd.read_parquet(p).gloss.tolist())
cov = pd.DataFrame(cov).T
cov.drop(columns="top_fingerspelled").to_csv(OUT / "coverage.csv")
with pd.option_context("display.max_colwidth", 90):
    display(cov[["units", "sign", "wlasl", "gislr", "fingerspell", "sentences_all_signs", "sentences_all_gislr"]]
            .astype(float).round(3))
    display(cov[["top_fingerspelled"]])

units   sign  wlasl  gislr  fingerspell  \
team30 rules_v1             143.0  0.958  0.951  0.448        0.042   
       rules_v2             150.0  0.973  0.973  0.547        0.027   
       hybrid_team          143.0  0.958  0.951  0.448        0.042   
       hybrid_guarded       150.0  0.973  0.973  0.547        0.027   
       t5_recorded          153.0  0.948  0.948  0.562        0.052   
       guard_on_recorded    154.0  0.955  0.955  0.532        0.045   
       reference_draft      150.0  0.980  0.980  0.560        0.020   
aslg   hybrid_guarded     17170.0  0.610  0.600  0.134        0.390   
       hybrid_team        15530.0  0.589  0.579  0.109        0.411   
       rules_v1           15530.0  0.589  0.579  0.109        0.411   
       rules_v2           17170.0  0.610  0.600  0.134        0.390   

                          sentences_all_signs  sentences_all_gislr  
team30 rules_v1                         0.833                0.000  
       rules_v2                         0.900                0.067  
       hybrid_team                      0.833                0.000  
       hybrid_guarded                   0.900                0.067  
       t5_recorded                      0.767                0.067  
       guard_on_recorded                0.800                0.033  
       reference_draft                  0.933                0.067  
aslg   hybrid_guarded                   0.062                0.000  
       hybrid_team                      0.074                0.000  
       rules_v1                         0.074                0.000  
       rules_v2                         0.062                0.000

top_fingerspelled
team30 rules_v1                      [(JOHN, 1), (MARKET, 1), (O'CLOCK, 1), (DHAKA, 1), (IMMEDIATELY, 1), (YET, 1)]
       rules_v2                                              [(JOHN, 1), (MARKET, 1), (DHAKA, 1), (IMMEDIATELY, 1)]
       hybrid_team                   [(JOHN, 1), (MARKET, 1), (O'CLOCK, 1), (DHAKA, 1), (IMMEDIATELY, 1), (YET, 1)]
       hybrid_guarded                                        [(JOHN, 1), (MARKET, 1), (DHAKA, 1), (IMMEDIATELY, 1)]
       t5_recorded            [(DO, 2), (JOHN, 1), (MARKET, 1), (O'CLOCK, 1), (DHAKA, 1), (IMMEDIATE, 1), (YET, 1)]
       guard_on_recorded                [(DO, 2), (JOHN, 1), (MARKET, 1), (O'CLOCK, 1), (DHAKA, 1), (IMMEDIATE, 1)]
       reference_draft                                                         [(JOHN, 1), (MARKET, 1), (DHAKA, 1)]
aslg   hybrid_guarded     [(EUROPEAN, 165), (WOULD, 87), (THESE, 76), (COMMISSION, 74), (DO, 67), (PARLIAMENT, 6...
       hybrid_team        [(EUROPEAN, 167), (COMMISSION, 74), (PARLIAMENT, 66), (EU, 65), (UNION, 64), (MR, 64),...
       rules_v1           [(EUROPEAN, 167), (COMMISSION, 74), (PARLIAMENT, 66), (EU, 65), (UNION, 64), (MR, 64),...
       rules_v2           [(EUROPEAN, 165), (WOULD, 87), (THESE, 76), (COMMISSION, 74), (DO, 67), (PARLIAMENT, 6...